# Песочница паттернов GoF: Strategy, Factory Method, Observer

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java. Каждый пример представлен **одним файлом** `.java` и запускается без отдельной компиляции командой `java Файл.java` (запуск исходного файла, Java 11 и выше).

**Идея задания.** Паттерн изучается не по описанию, а по разнице: сначала запускается код «до» — работающий, но жёстко связанный; затем тот же сценарий переписывается на паттерн, и проверки доказывают, что **поведение не изменилось**, а **новое требование добавляется без правки старого кода**.

**Состав работы**

| Раздел | Паттерн | Что делает студент |
|---|---|---|
| 2 | Strategy | **образец** — разобран полностью, нужно только запустить и разобраться |
| 3 | Factory Method | задача: самостоятельно переписать код «до» |
| 4 | Observer | задача: самостоятельно переписать код «до» |
| 5–6 | — | сравнение, выводы, контрольные вопросы |

**Правила оформления, от которых зависит автопроверка**

1. Имена файлов строго: `TariffAfter.java`, `FactoryAfter.java`, `ObserverAfter.java`.
2. В каждом файле **первым** идёт открытый класс с методом `main` — иначе запуск исходного файла не работает.
3. Каждый файл сам себя проверяет: печатает строки `[OK]` / `[FAIL]` и бросает `AssertionError`, если хотя бы одна проверка не пройдена.
4. Идентификаторы — латиницей и описательно (`tariffStrategy`, а не `ts`), комментарии — по-русски.
5. К каждому паттерну — диаграмма классов в файле `*.puml`.

**Сдача.** Все файлы `.java`, `.puml`, сам блокнот и вывод ячейки самопроверки — в pull request.

## 1. Подготовка среды

In [ ]:
# 1.1 Проверить, какая Java доступна в среде.
# Нужна версия 11 или выше: только с неё работает запуск одного исходного файла (java Файл.java).
!java -version
!javac -version

Если версия ниже 11, запуск одного исходного файла недоступен — всюду в блокноте вместо `!java Файл.java` используйте два шага: `!javac Файл.java && java Файл`.

Проверка «живого» запуска (ячейка 1.2 записывает файл, 1.3 — запускает):

In [ ]:
%%writefile Hello.java
public class Hello {
    public static void main(String[] args) {
        System.out.println("Среда готова, Java работает: " + System.getProperty("java.version"));
    }
}

In [ ]:
!java Hello.java

---

## 2. Образец: Strategy (разобран полностью)

**Сценарий.** Расчёт платы за перевозку груза. Плата зависит от вида отправки: повагонная, контейнерная, маршрутная. Формулы учебные.

### 2.1 Код «до»: один метод знает про все виды отправок

In [ ]:
%%writefile TariffBefore.java
// Код «до»: вид отправки разбирается условным оператором внутри одного метода.
public class TariffBefore {

    public static void main(String[] args) {
        System.out.printf("повагонная   : %10.2f%n", cost("WAGON", 60, 500));
        System.out.printf("контейнерная : %10.2f%n", cost("CONTAINER", 24, 500));
        System.out.printf("маршрутная   : %10.2f%n", cost("BLOCK_TRAIN", 3000, 500));
        System.out.printf("рефрижератор : %10.2f%n", cost("REFRIGERATED", 60, 500));
    }

    static double cost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;   // про новый вид отправки метод не знает
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

In [ ]:
!java TariffBefore.java

### 2.2 Разбор: что именно плохо

Запустив код «до», обратите внимание:

* рефрижераторная отправка дала **0.00** — метод о ней не знает, и единственный способ это исправить — залезть внутрь уже работающего метода;
* каждый новый вид отправки удлиняет один и тот же метод, то есть нарушает **принцип открытости/закрытости** (открыт для расширения, закрыт для изменения);
* метод одновременно отвечает за выбор варианта и за сам расчёт — нарушение **единственной ответственности**;
* протестировать один вид отправки отдельно от остальных нельзя.

**Что делает Strategy.** Каждый вариант расчёта становится отдельным классом с общим интерфейсом, а вызывающий код работает с интерфейсом и больше не знает перечня вариантов.

### 2.3 Код «после»: вариант расчёта вынесен в стратегию

In [ ]:
%%writefile TariffAfter.java
// Код «после»: каждый вид отправки — отдельная стратегия расчёта.
import java.util.LinkedHashMap;
import java.util.Map;

public class TariffAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог стратегий: вызывающий код собирает его, но не содержит формул
        Map<String, TariffStrategy> tariffCatalog = new LinkedHashMap<>();
        tariffCatalog.put("WAGON", new WagonTariff());
        tariffCatalog.put("CONTAINER", new ContainerTariff());
        tariffCatalog.put("BLOCK_TRAIN", new BlockTrainTariff());

        TariffCalculator tariffCalculator = new TariffCalculator();

        System.out.println("--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---");
        for (Map.Entry<String, TariffStrategy> catalogEntry : tariffCatalog.entrySet()) {
            double calculatedCost = tariffCalculator.calculate(catalogEntry.getValue(), 60, 500);
            System.out.printf("%-12s %-28s %10.2f%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), calculatedCost);
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("повагонная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("WAGON"), 60, 500),
                           legacyCost("WAGON", 60, 500)));
        check("контейнерная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("CONTAINER"), 24, 500),
                           legacyCost("CONTAINER", 24, 500)));
        check("маршрутная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("BLOCK_TRAIN"), 3000, 500),
                           legacyCost("BLOCK_TRAIN", 3000, 500)));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---");
        TariffStrategy refrigeratedTariff = new TariffStrategy() {
            public double cost(double massTons, int distanceKm) {
                return 1.80 * massTons * distanceKm / 100.0 + 600.0;
            }
            public String title() {
                return "рефрижераторная отправка";
            }
        };
        double refrigeratedCost = tariffCalculator.calculate(refrigeratedTariff, 60, 500);
        System.out.printf("%-12s %-28s %10.2f%n", "REFRIGERATED", refrigeratedTariff.title(), refrigeratedCost);
        check("новая стратегия рассчитана, класс TariffCalculator не изменён", refrigeratedCost > 0.0);
        check("в коде «до» тот же вид отправки давал нуль", legacyCost("REFRIGERATED", 60, 500) == 0.0);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // --- служебное ---

    static boolean equalMoney(double firstValue, double secondValue) {
        return Math.abs(firstValue - secondValue) < 0.01;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Старая реализация сохранена только как эталон поведения для характеризующих тестов
    static double legacyCost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

// --- участники паттерна Strategy ---

/** Стратегия: общий интерфейс всех вариантов расчёта. */
interface TariffStrategy {
    double cost(double massTons, int distanceKm);
    String title();
}

/** Контекст: умеет считать, но не знает, какие бывают виды отправок. */
class TariffCalculator {
    double calculate(TariffStrategy tariffStrategy, double massTons, int distanceKm) {
        double calculatedValue = tariffStrategy.cost(massTons, distanceKm);
        return Math.round(calculatedValue * 100.0) / 100.0;
    }
}

class WagonTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 1.20 * massTons * distanceKm / 100.0 + 150.0;
    }
    public String title() {
        return "повагонная отправка";
    }
}

class ContainerTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.90 * massTons * distanceKm / 100.0 + 400.0;
    }
    public String title() {
        return "контейнерная отправка";
    }
}

class BlockTrainTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.70 * massTons * distanceKm / 100.0;
    }
    public String title() {
        return "маршрутная отправка";
    }
}

In [ ]:
!java TariffAfter.java

### 2.4 Диаграмма классов образца

В Colab нет команды `plantuml`. Ячейка ниже один раз ставит Java, если её нет, и скачивает `plantuml.jar`. Дальше диаграммы строятся командой `java -jar plantuml.jar`.


In [ ]:
import pathlib
import shutil
import subprocess
import urllib.request

if shutil.which("java") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"])

jar = pathlib.Path("plantuml.jar")
if not jar.exists() or jar.stat().st_size < 1_000_000:
    urllib.request.urlretrieve(
        "https://github.com/plantuml/plantuml/releases/download/v1.2024.8/plantuml-1.2024.8.jar",
        jar,
    )
print(subprocess.check_output(["java", "-jar", str(jar), "-version"], text=True))


In [ ]:
%%writefile strategy.puml
@startuml
title Strategy: расчёт платы за перевозку

interface TariffStrategy {
  +cost(massTons, distanceKm) : double
  +title() : String
}

class TariffCalculator {
  +calculate(tariffStrategy, massTons, distanceKm) : double
}

class WagonTariff
class ContainerTariff
class BlockTrainTariff

TariffCalculator o--> TariffStrategy : использует
TariffStrategy <|.. WagonTariff
TariffStrategy <|.. ContainerTariff
TariffStrategy <|.. BlockTrainTariff
@enduml

In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

diagram = "strategy.puml"
completed = subprocess.run(
    ["java", "-jar", "plantuml.jar", "-tpng", diagram],
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or "PlantUML не построил диаграмму")
png = Path(diagram).with_suffix(".png")
if not png.exists():
    raise FileNotFoundError(completed.stderr or str(png))
display(Image(filename=str(png)))


### 2.5 Вопросы к образцу (ответить текстом в этой ячейке)

1. Какие классы пришлось бы изменить в коде «до» и в коде «после», чтобы добавить рефрижераторную отправку?
   **Ответ:** В коде «до» правится единственный метод `cost` класса `TariffBefore`: в цепочку `if` добавляется ветка `REFRIGERATED`. В коде «после» `TariffCalculator`, `WagonTariff`, `ContainerTariff` и `BlockTrainTariff` не меняются. Добавляется новая стратегия (в образце она задана анонимным классом) и одна строка её регистрации в каталоге в `main`.
2. Зачем в файле оставлен метод `legacyCost`, если он больше не используется в расчёте?
   **Ответ:** Это эталон прежнего поведения для характеризующих тестов. Проверки сравнивают сумму стратегии с `legacyCost` и показывают, что перенос формул в отдельные классы не изменил результат. В рабочем расчёте метод не участвует.
3. Что произойдёт, если в каталоге не окажется запрошенного вида отправки, и как это следует обработать?
   **Ответ:** `tariffCatalog.get(...)` вернёт `null`, и `calculate` упадёт с `NullPointerException`. Перед расчётом нужно убедиться, что стратегия найдена, и сообщить о неизвестном виде отправки, а не вызывать метод у пустой ссылки.


---

## 3. Задача 1: Factory Method

**Сценарий.** Описание вагона по его типу: крытый, платформа, цистерна. Ниже — код «до», построенный на той же ошибке: условный оператор вместо полиморфизма.

### 3.1 Код «до» (запустить, не изменяя)

In [ ]:
%%writefile FactoryBefore.java
// Код «до»: создание и описание объекта подменены разбором строкового типа.
public class FactoryBefore {

    public static void main(String[] args) {
        System.out.println(describe("COVERED", 68));
        System.out.println(describe("PLATFORM", 71));
        System.out.println(describe("TANK", 66));
        System.out.println(describe("HOPPER", 70));   // новый тип — обработчика нет
    }

    static String describe(String wagonType, double capacityTons) {
        if (wagonType.equals("COVERED")) {
            return "Крытый вагон: грузоподъёмность " + capacityTons + " т, тарные и штучные грузы";
        } else if (wagonType.equals("PLATFORM")) {
            return "Платформа: грузоподъёмность " + capacityTons + " т, длинномерные и тяжеловесные грузы";
        } else if (wagonType.equals("TANK")) {
            return "Цистерна: грузоподъёмность " + capacityTons + " т, наливные грузы";
        }
        return "Неизвестный тип вагона: " + wagonType;
    }
}

In [ ]:
!java FactoryBefore.java

### 3.2 Задание

Переписать сценарий на **Factory Method** в файле `FactoryAfter.java`. Требования:

1. Общий тип продукта: интерфейс или абстрактный класс `Wagon` с методами `description()` и `capacityTons()`.
2. Не менее трёх конкретных продуктов: `CoveredWagon`, `Platform`, `TankWagon`.
3. Создатель с фабричным методом: абстрактный класс или интерфейс `WagonCreator` с методом `createWagon(double capacityTons)` и конкретные создатели для каждого типа.
4. Вызывающий код (`main`) работает **только** с типами `WagonCreator` и `Wagon` и не содержит ни одного `if` по типу вагона.
5. Четвёртый тип (`HopperWagon`, зерновоз) добавляется **новым классом**, без правки ранее написанных классов; это должно быть видно по коду.
6. Проверки в файле (метод `check`, как в образце), минимум четыре:
   * описание каждого из трёх типов не пустое и содержит грузоподъёмность;
   * тип созданного объекта соответствует ожидаемому создателю;
   * четвёртый тип работает наравне с остальными;
   * метод `main` не содержит разбора типов (проверяется глазами преподавателя, в код выносить не нужно).
7. При неуспешной проверке — `throw new AssertionError(...)`.

**Подсказка по структуре файла:** первым идёт `public class FactoryAfter` с `main`, после него — интерфейсы и классы паттерна (как в `TariffAfter.java`).

In [ ]:
%%writefile FactoryAfter.java
// Вариант 12, энергодиспетчерский пункт.
// Тип потребителя создаётся фабричным методом, а не условным оператором.
public class FactoryAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        ConsumerCreator tractionCreator = new TractionConsumerCreator();
        ConsumerCreator signalingCreator = new SignalingConsumerCreator();
        ConsumerCreator industrialCreator = new IndustrialConsumerCreator();
        ConsumerCreator householdCreator = new HouseholdConsumerCreator();

        EnergyConsumer tractionConsumer = tractionCreator.createConsumer(2500);
        EnergyConsumer signalingConsumer = signalingCreator.createConsumer(80);
        EnergyConsumer industrialConsumer = industrialCreator.createConsumer(1200);
        EnergyConsumer householdConsumer = householdCreator.createConsumer(150);

        System.out.println(tractionConsumer.description());
        System.out.println(signalingConsumer.description());
        System.out.println(industrialConsumer.description());
        System.out.println(householdConsumer.description());
        System.out.println();

        check("тяговый потребитель: описание не пустое и содержит мощность",
                describedWithPower(tractionConsumer));
        check("потребитель СЦБ: описание не пустое и содержит мощность",
                describedWithPower(signalingConsumer));
        check("промышленный потребитель: описание не пустое и содержит мощность",
                describedWithPower(industrialConsumer));
        check("тяговый потребитель создан своим создателем",
                tractionConsumer instanceof TractionConsumer);
        check("потребитель СЦБ создан своим создателем",
                signalingConsumer instanceof SignalingConsumer);
        check("промышленный потребитель создан своим создателем",
                industrialConsumer instanceof IndustrialConsumer);
        check("бытовой потребитель добавлен новым классом и работает наравне с остальными",
                householdConsumer instanceof HouseholdConsumer && describedWithPower(householdConsumer));

        ensureAllPassed();
    }

    static boolean describedWithPower(EnergyConsumer consumer) {
        return !consumer.description().isEmpty()
                && consumer.description().contains(String.valueOf(consumer.contractedPowerKw()));
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static void ensureAllPassed() {
        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }
}

/** Продукт: потребитель энергии. Конкретный тип вызывающему коду неизвестен. */
interface EnergyConsumer {
    String description();
    double contractedPowerKw();
}

/** Создатель: фабричный метод выбирает класс потребителя. */
interface ConsumerCreator {
    EnergyConsumer createConsumer(double contractedPowerKw);
}

class TractionConsumer implements EnergyConsumer {
    private final double contractedPowerKw;

    TractionConsumer(double contractedPowerKw) {
        this.contractedPowerKw = contractedPowerKw;
    }

    public String description() {
        return "Тяговый потребитель: договорная мощность " + contractedPowerKw
                + " кВт, электроподвижной состав";
    }

    public double contractedPowerKw() {
        return contractedPowerKw;
    }
}

class SignalingConsumer implements EnergyConsumer {
    private final double contractedPowerKw;

    SignalingConsumer(double contractedPowerKw) {
        this.contractedPowerKw = contractedPowerKw;
    }

    public String description() {
        return "Потребитель СЦБ: договорная мощность " + contractedPowerKw
                + " кВт, устройства сигнализации и связи";
    }

    public double contractedPowerKw() {
        return contractedPowerKw;
    }
}

class IndustrialConsumer implements EnergyConsumer {
    private final double contractedPowerKw;

    IndustrialConsumer(double contractedPowerKw) {
        this.contractedPowerKw = contractedPowerKw;
    }

    public String description() {
        return "Промышленный потребитель: договорная мощность " + contractedPowerKw
                + " кВт, производственная нагрузка";
    }

    public double contractedPowerKw() {
        return contractedPowerKw;
    }
}

/** Четвёртый тип: добавлен новым классом, прежние продукты и создатели не менялись. */
class HouseholdConsumer implements EnergyConsumer {
    private final double contractedPowerKw;

    HouseholdConsumer(double contractedPowerKw) {
        this.contractedPowerKw = contractedPowerKw;
    }

    public String description() {
        return "Бытовой потребитель: договорная мощность " + contractedPowerKw
                + " кВт, освещение и бытовая нагрузка";
    }

    public double contractedPowerKw() {
        return contractedPowerKw;
    }
}

class TractionConsumerCreator implements ConsumerCreator {
    public EnergyConsumer createConsumer(double contractedPowerKw) {
        return new TractionConsumer(contractedPowerKw);
    }
}

class SignalingConsumerCreator implements ConsumerCreator {
    public EnergyConsumer createConsumer(double contractedPowerKw) {
        return new SignalingConsumer(contractedPowerKw);
    }
}

class IndustrialConsumerCreator implements ConsumerCreator {
    public EnergyConsumer createConsumer(double contractedPowerKw) {
        return new IndustrialConsumer(contractedPowerKw);
    }
}

class HouseholdConsumerCreator implements ConsumerCreator {
    public EnergyConsumer createConsumer(double contractedPowerKw) {
        return new HouseholdConsumer(contractedPowerKw);
    }
}


In [ ]:
!java FactoryAfter.java

In [ ]:
%%writefile factory.puml
@startuml
title Factory Method: тип потребителя энергодиспетчерского пункта

interface EnergyConsumer {
  +description() : String
  +contractedPowerKw() : double
}

interface ConsumerCreator {
  +createConsumer(contractedPowerKw) : EnergyConsumer
}

class TractionConsumer
class SignalingConsumer
class IndustrialConsumer
class HouseholdConsumer

class TractionConsumerCreator
class SignalingConsumerCreator
class IndustrialConsumerCreator
class HouseholdConsumerCreator

EnergyConsumer <|.. TractionConsumer
EnergyConsumer <|.. SignalingConsumer
EnergyConsumer <|.. IndustrialConsumer
EnergyConsumer <|.. HouseholdConsumer

ConsumerCreator <|.. TractionConsumerCreator
ConsumerCreator <|.. SignalingConsumerCreator
ConsumerCreator <|.. IndustrialConsumerCreator
ConsumerCreator <|.. HouseholdConsumerCreator

TractionConsumerCreator ..> TractionConsumer : создаёт
SignalingConsumerCreator ..> SignalingConsumer : создаёт
IndustrialConsumerCreator ..> IndustrialConsumer : создаёт
HouseholdConsumerCreator ..> HouseholdConsumer : создаёт
@enduml


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

diagram = "factory.puml"
completed = subprocess.run(
    ["java", "-jar", "plantuml.jar", "-tpng", diagram],
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or "PlantUML не построил диаграмму")
png = Path(diagram).with_suffix(".png")
if not png.exists():
    raise FileNotFoundError(completed.stderr or str(png))
display(Image(filename=str(png)))


**Вывод по задаче 1** (чем создание через фабричный метод отличается от создания через `new` в вызывающем коде, и когда фабрика излишня): вариант 12, энергодиспетчерский пункт. Вызывающий код зависит только от `ConsumerCreator` и `EnergyConsumer`. Он не выбирает конкретный класс через `new` и не разбирает тип потребителя условным оператором. Какой потребитель появится, решает создатель: тяговый, СЦБ, промышленный или бытовой. Бытовой потребитель добавлен парой классов `HouseholdConsumer` и `HouseholdConsumerCreator`, ранее написанные классы не менялись. Фабрика излишня, когда тип потребителя один и нового вида не ожидается: интерфейс и создатель тогда только окружают единственный `new`.


---

## 4. Задача 2: Observer

**Сценарий.** Станция фиксирует прибытие поезда и должна оповестить получателей: табло, журнал, рассылку. В коде «до» станция сама знает про всех получателей.

### 4.1 Код «до» (запустить, не изменяя)

In [ ]:
%%writefile ObserverBefore.java
// Код «до»: источник события жёстко связан со всеми получателями.
public class ObserverBefore {

    public static void main(String[] args) {
        Station station = new Station("Гомель");
        station.trainArrived("3022");
        station.trainArrived("6701");
    }
}

class Station {

    private final String stationName;

    Station(String stationName) {
        this.stationName = stationName;
    }

    void trainArrived(String trainNumber) {
        // Любой новый получатель требует правки этого метода
        System.out.println("[табло]  Поезд " + trainNumber + " прибыл на станцию " + stationName);
        System.out.println("[журнал] " + stationName + ": зафиксировано прибытие поезда " + trainNumber);
    }
}

In [ ]:
!java ObserverBefore.java

### 4.2 Задание

Переписать сценарий на **Observer** в файле `ObserverAfter.java`. Требования:

1. Интерфейс наблюдателя `ArrivalListener` с методом `onTrainArrived(String stationName, String trainNumber)`.
2. Субъект `Station` с методами `addListener`, `removeListener`, `trainArrived`; класс `Station` **не знает**, какие именно наблюдатели существуют.
3. Не менее трёх наблюдателей: табло, журнал, рассылка. Журнал должен **накапливать** записи, чтобы их можно было проверить.
4. Четвёртый наблюдатель (например, передача в АСУ) добавляется без правки класса `Station`.
5. Проверки в файле, минимум четыре:
   * после одного события журнал получил ровно одну запись;
   * после отписки наблюдателя его записи больше не появляются;
   * все подписчики получили одно и то же событие;
   * добавление четвёртого наблюдателя не потребовало изменения `Station`.
6. В выводе указать порядок получения события подписчиками и ответить на вопрос в выводе ниже.

**Подсказка:** список наблюдателей удобно хранить в `java.util.ArrayList`, обход — обычным `for`.

In [ ]:
%%writefile ObserverAfter.java
// Вариант 12, энергодиспетчерский пункт.
// Подстанция рассылает событие аварии подписчикам и не знает, кто они.
import java.util.ArrayList;
import java.util.List;

public class ObserverAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        Substation substation = new Substation("Гомель-тяговая");
        DispatcherBoardListener boardListener = new DispatcherBoardListener();
        AccidentJournalListener journalListener = new AccidentJournalListener();
        CrewMailingListener mailingListener = new CrewMailingListener();

        substation.addListener(boardListener);
        substation.addListener(journalListener);
        substation.addListener(mailingListener);

        System.out.println("--- Авария КЗ-1: порядок подписки — табло, журнал, рассылка ---");
        substation.accidentOccurred("КЗ-1");

        check("после одной аварии журнал получил ровно одну запись",
                journalListener.recordCount() == 1);
        check("табло, журнал и рассылка получили одно и то же событие",
                sameAccident(boardListener, journalListener, mailingListener, "Гомель-тяговая", "КЗ-1"));

        substation.removeListener(journalListener);
        System.out.println();
        System.out.println("--- Авария КЗ-2: журнал отписан ---");
        substation.accidentOccurred("КЗ-2");
        check("после отписки журнал больше не получает записи",
                journalListener.recordCount() == 1);
        check("остальные подписчики аварию КЗ-2 получили",
                boardListener.lastAccidentCode().equals("КЗ-2")
                        && mailingListener.lastAccidentCode().equals("КЗ-2"));

        AsuListener asuListener = new AsuListener();
        substation.addListener(asuListener);
        System.out.println();
        System.out.println("--- Авария ОЗЗ-3: четвёртый наблюдатель добавлен без правки Substation ---");
        substation.accidentOccurred("ОЗЗ-3");
        check("передача в АСУ получила событие, класс Substation не менялся",
                asuListener.receivedCount() == 1
                        && asuListener.lastAccidentCode().equals("ОЗЗ-3"));

        System.out.println();
        System.out.println("Порядок уведомления равен порядку подписки: табло, журнал, рассылка; АСУ — после отдельной подписки.");

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    static boolean sameAccident(DispatcherBoardListener boardListener,
                                AccidentJournalListener journalListener,
                                CrewMailingListener mailingListener,
                                String substationName,
                                String accidentCode) {
        return boardListener.lastSubstationName().equals(substationName)
                && journalListener.lastSubstationName().equals(substationName)
                && mailingListener.lastSubstationName().equals(substationName)
                && boardListener.lastAccidentCode().equals(accidentCode)
                && journalListener.lastAccidentCode().equals(accidentCode)
                && mailingListener.lastAccidentCode().equals(accidentCode);
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }
}

/** Наблюдатель: получатель сообщения об аварии на подстанции. */
interface AccidentListener {
    void onAccident(String substationName, String accidentCode);
}

/** Субъект: хранит подписчиков и не знает их конкретных классов. */
class Substation {

    private final String substationName;
    private final List<AccidentListener> listeners = new ArrayList<>();

    Substation(String substationName) {
        this.substationName = substationName;
    }

    void addListener(AccidentListener listener) {
        listeners.add(listener);
    }

    void removeListener(AccidentListener listener) {
        listeners.remove(listener);
    }

    void accidentOccurred(String accidentCode) {
        for (AccidentListener listener : listeners) {
            listener.onAccident(substationName, accidentCode);
        }
    }
}

class DispatcherBoardListener implements AccidentListener {
    private String lastSubstationName = "";
    private String lastAccidentCode = "";

    public void onAccident(String substationName, String accidentCode) {
        lastSubstationName = substationName;
        lastAccidentCode = accidentCode;
        System.out.println("[табло]  Авария " + accidentCode + " на подстанции " + substationName);
    }

    String lastSubstationName() {
        return lastSubstationName;
    }

    String lastAccidentCode() {
        return lastAccidentCode;
    }
}

class AccidentJournalListener implements AccidentListener {
    private final List<String> records = new ArrayList<>();
    private String lastSubstationName = "";
    private String lastAccidentCode = "";

    public void onAccident(String substationName, String accidentCode) {
        lastSubstationName = substationName;
        lastAccidentCode = accidentCode;
        records.add(substationName + ": зафиксирована авария " + accidentCode);
        System.out.println("[журнал] " + records.get(records.size() - 1));
    }

    int recordCount() {
        return records.size();
    }

    String lastSubstationName() {
        return lastSubstationName;
    }

    String lastAccidentCode() {
        return lastAccidentCode;
    }
}

class CrewMailingListener implements AccidentListener {
    private String lastSubstationName = "";
    private String lastAccidentCode = "";

    public void onAccident(String substationName, String accidentCode) {
        lastSubstationName = substationName;
        lastAccidentCode = accidentCode;
        System.out.println("[рассылка] Подстанция " + substationName + ", авария " + accidentCode);
    }

    String lastSubstationName() {
        return lastSubstationName;
    }

    String lastAccidentCode() {
        return lastAccidentCode;
    }
}

/** Четвёртый наблюдатель: подключается через addListener, класс Substation не меняется. */
class AsuListener implements AccidentListener {
    private String lastAccidentCode = "";
    private int receivedCount = 0;

    public void onAccident(String substationName, String accidentCode) {
        lastAccidentCode = accidentCode;
        receivedCount++;
        System.out.println("[АСУ]    Передана авария " + accidentCode + " на подстанции " + substationName);
    }

    String lastAccidentCode() {
        return lastAccidentCode;
    }

    int receivedCount() {
        return receivedCount;
    }
}


In [ ]:
!java ObserverAfter.java

In [ ]:
%%writefile observer.puml
@startuml
title Observer: авария на подстанции

interface AccidentListener {
  +onAccident(substationName, accidentCode)
}

class Substation {
  +addListener(listener)
  +removeListener(listener)
  +accidentOccurred(accidentCode)
}

class DispatcherBoardListener
class AccidentJournalListener
class CrewMailingListener
class AsuListener

Substation o--> "0..*" AccidentListener : оповещает
AccidentListener <|.. DispatcherBoardListener
AccidentListener <|.. AccidentJournalListener
AccidentListener <|.. CrewMailingListener
AccidentListener <|.. AsuListener
@enduml


In [ ]:
import subprocess
from pathlib import Path
from IPython.display import Image, display

diagram = "observer.puml"
completed = subprocess.run(
    ["java", "-jar", "plantuml.jar", "-tpng", diagram],
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or "PlantUML не построил диаграмму")
png = Path(diagram).with_suffix(".png")
if not png.exists():
    raise FileNotFoundError(completed.stderr or str(png))
display(Image(filename=str(png)))


**Вывод по задаче 2** (какую связанность снял Observer и какие новые проблемы он создал — порядок уведомлений, ошибка в одном подписчике, забытая отписка): вариант 12. Подстанция больше не перечисляет табло диспетчера, журнал аварий и рассылку бригадам внутри метода аварии. Она хранит список `AccidentListener` и вызывает у каждого один метод. Передача в АСУ подключена через `addListener`, класс `Substation` для этого не менялся. Порядок сообщений равен порядку подписки, а не порядку важности получателей. Если один подписчик бросит исключение, цикл прервётся и следующие об аварии не узнают. Если получателя не снять через `removeListener`, подстанция продолжит его удерживать: объект не будет собран, а отписанный по смыслу получатель всё ещё будет получать события.


---

## 5. Сравнение и выводы

Вариант 12 — энергодиспетчерский пункт. Образец Strategy в разделе 2 общий. Задачи Factory Method и Observer выполнены на этой предметной области: фабрика создаёт тип потребителя, наблюдатели получают аварию на подстанции. Тарифный план «день / ночь» — место для Strategy в том же пункте.

### 5.1 Итоговая таблица

| Паттерн | Какую проблему кода «до» устраняет | Что добавляется без правки старого кода | Чем приходится заплатить | Где применим в вашем проекте |
|---|---|---|---|---|
| Strategy | Один метод одновременно выбирает вариант и выполняет расчёт. Новый вариант требует правки этого метода | Новый класс стратегии. Контекст и старые стратегии не меняются | Больше классов. Вызывающий код должен явно выбрать стратегию | Энергодиспетчерский пункт: тарифный план «день» и «ночь». Новый план — новая стратегия, расчёт платы потребителю не меняет диспетчерский код |
| Factory Method | Тип потребителя выбирается цепочкой условий, неизвестный тип даёт заглушку | Новая пара «потребитель + создатель». Вызывающий код работает с `EnergyConsumer` и `ConsumerCreator` | Две параллельные иерархии вместо одного метода. Фабрика не нужна, если тип один | Энергодиспетчерский пункт: тяговый, СЦБ, промышленный и бытовой потребители. Новый тип добавляется своим создателем |
| Observer | Источник события сам знает всех получателей и правится при каждом новом | Новый подписчик через `addListener`. Класс `Substation` не знает конкретных наблюдателей | Порядок равен порядку подписки. Исключение в одном подписчике обрывает остальных. Забытая отписка удерживает объект | Энергодиспетчерский пункт: авария на подстанции оповещает табло, журнал, бригады и АСУ. Подстанция не вызывает их напрямую |

### 5.2 Контрольные вопросы

1. Какой принцип SOLID нарушает разбор типа условным оператором и как каждый из трёх паттернов его восстанавливает?
   **Ответ:** Нарушается принцип открытости/закрытости: работающий код приходится менять при каждом новом варианте. Заодно метод берёт на себя и выбор варианта, и само действие, то есть нарушается единственная ответственность. Strategy выносит каждый тарифный план в свой класс. Factory Method выносит создание каждого типа потребителя в свой создатель. Observer выносит каждого получателя аварии в свой подписчик. Старый код закрыт для правки и открыт для нового класса.
2. Чем Factory Method отличается от статического метода создания (`static factory method`) и от Abstract Factory?
   **Ответ:** Фабричный метод полиморфен: подкласс создателя решает, какой класс потребителя создавать, и вызывающий код этого класса не называет. Статический метод создания — обычный метод класса, он часто сам содержит разбор типа и возвращает экземпляр, не подменяя иерархию. Abstract Factory создаёт семейство связанных продуктов, а Factory Method — один продукт.
3. Strategy и State имеют почти одинаковую структуру классов. В чём разница по назначению?
   **Ответ:** В Strategy вызывающий код выбирает алгоритм и может заменить его, чтобы посчитать иначе: днём одна ставка, ночью другая. В State объект меняет поведение сам, когда меняется его внутреннее состояние. Клиент не выбирает «режим» при каждом вызове: состояние переключается по правилам самого объекта.
4. Почему Observer может привести к утечке памяти и как этого избежать?
   **Ответ:** Субъект хранит ссылку на подписчика. Пока ссылка есть, сборщик мусора подписчика не заберёт, даже если остальная программа его уже не использует. Избегается вызовом `removeListener`, когда получатель больше не нужен. Забытая отписка в этой работе как раз оставляет журнал аварий в списке подстанции.
5. Приведите пример, когда введение паттерна **вредно**. Что такое преждевременное усложнение?
   **Ответ:** Если у пункта один тип потребителя и второго не ожидается, интерфейс фабрики и отдельный создатель вокруг одного `new` только мешают читать код. Преждевременное усложнение — это ввод паттерна до появления второй реализации: структура растёт, а вариативности, ради которой она нужна, ещё нет.

### 5.3 Три вывода по работе в целом

1. Во всех трёх задачах код «до» разбирает варианты условным оператором. Паттерн заменяет эту ветку новым классом, и прежний код не правится.
2. Проверки с `[OK]` фиксируют поведение варианта 12: бытовой потребитель появляется без изменения уже написанных классов, отписка журнала аварий останавливает его записи, АСУ получает аварию без правки `Substation`.
3. Плата за это — дополнительные типы и новые обязанности. Стратегию тарифа нужно выбрать, тип потребителя связать с создателем, а подписчика аварии снять, иначе останется и лишний вызов, и ссылка в памяти.


---

## 6. Варианты предметных областей

Номер варианта — по номеру студента в списке группы. Образец (раздел 2) выполняется всеми одинаково; задачи 1 и 2 — на своей предметной области.

| № | Предметная область | Фабрика создаёт | Стратегия варьирует | Наблюдатели получают событие |
|---|---|---|---|---|
| 1 | Грузовая станция | тип вагона | расчёт платы за пользование вагонами | вагон подан под погрузку |
| 2 | Локомотивное депо | вид ремонта (ТО-2, ТР-1, ТР-2) | приоритет заявок на ремонт | локомотив выдан из ремонта |
| 3 | Пассажирская касса | тип проездного документа | начисление скидки | рейс задержан |
| 4 | Контейнерный терминал | тип контейнера | выбор места на площадке | контейнер прибыл |
| 5 | Автотранспортное предприятие | тип транспортного средства | расчёт расхода топлива | путевой лист закрыт |
| 6 | Склад | тип тары | правило размещения (FIFO, ABC) | запас ниже нормы |
| 7 | Сортировочная станция | тип отцепа | очерёдность расформирования | состав расформирован |
| 8 | Пригородные перевозки | тип абонемента | тарификация по зонам | изменено расписание |
| 9 | Пункт таможенного оформления | тип декларации | выбор канала контроля | груз оформлен |
| 10 | Экспедиторская компания | тип заявки (авто, ж. д., море) | выбор перевозчика по критерию | статус отправки изменён |
| 11 | Путевое хозяйство | тип дефекта рельса | назначение вида работ | обнаружен дефект |
| 12 | Энергодиспетчерский пункт | тип потребителя | тарифный план (день, ночь) | авария на подстанции |
| 13 | Вагонное депо | тип неисправности | выбор отцепочного или безотцепочного ремонта | вагон забракован |
| 14 | Логистический центр | тип упаковки | правило консолидации отправок | сформирована отправка |
| 15 | Центр фирменного транспортного обслуживания | тип договора | расчёт скидки за объём | заявка согласована |

---

## 7. Чек-лист самопроверки

- [ ] Раздел 1 выполнен, версия Java не ниже 11
- [ ] `TariffBefore.java` и `TariffAfter.java` запущены, вывод сохранён
- [ ] Отвечены вопросы к образцу (п. 2.5)
- [ ] `FactoryAfter.java` написан, все проверки `[OK]`, в `main` нет разбора типов
- [ ] Четвёртый тип вагона добавлен новым классом
- [ ] `ObserverAfter.java` написан, все проверки `[OK]`, отписка работает
- [ ] Четвёртый наблюдатель добавлен без правки класса `Station`
- [ ] Построены три диаграммы: `strategy.puml`, `factory.puml`, `observer.puml`
- [ ] Заполнена итоговая таблица (п. 5.1)
- [ ] Отвечены пять контрольных вопросов и сформулированы три вывода
- [ ] Блокнот выполняется сверху вниз без ошибок после *Среда выполнения → Перезапустить и выполнить все*
- [ ] Запущена ячейка самопроверки, её вывод вставлен в описание pull request

## 8. Ячейка самопроверки

Ячейка сама компилирует и запускает все три файла, проверяет отсутствие `[FAIL]`, наличие диаграмм и минимальные признаки соблюдения требований. Запускать после того, как задачи выполнены.

In [ ]:
#@title Запустить самопроверку и получить отчёт
import os, re, subprocess, datetime

ФИО_исполнителя = "Кравченко А.С."  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
Номер_варианта = 12  #@param {type:"integer"}
ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ПРОГРАММЫ = [
    ("Образец, Strategy", "TariffAfter.java"),
    ("Задача 1, Factory Method", "FactoryAfter.java"),
    ("Задача 2, Observer", "ObserverAfter.java"),
]
ДИАГРАММЫ = ["strategy.puml", "factory.puml", "observer.puml"]

def запустить(имя_файла):
    """Запустить один исходный файл Java и вернуть (успех, пояснение)."""
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    if завершение.returncode != 0:
        значимые = [s for s in вывод.splitlines() if s.strip() and not s.startswith("Picked up")]
        первая_строка = значимые[0] if значимые else "без сообщения"
        return False, "аварийное завершение: " + первая_строка[:90]
    if "[FAIL]" in вывод:
        return False, "проверок не пройдено: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 3:
        return False, "проверок в файле только %d, требуется не менее 3" % успешных
    return True, "проверок пройдено: %d" % успешных

результаты = []
for название, файл in ПРОГРАММЫ:
    результаты.append((название, файл) + запустить(файл))

# Признаки соблюдения требований, которые можно проверить по тексту
замечания = []
def тело_метода(исходный_текст, сигнатура="main("):
    """Вернуть тело метода по сопоставлению фигурных скобок (без построчных комментариев)."""
    без_комментариев = re.sub(r"//[^\n]*", "", исходный_текст)
    начало = без_комментариев.find(сигнатура)
    if начало < 0:
        return ""
    открывающая = без_комментариев.find("{", начало)
    if открывающая < 0:
        return ""
    глубина = 0
    for позиция in range(открывающая, len(без_комментариев)):
        символ = без_комментариев[позиция]
        if символ == "{":
            глубина += 1
        elif символ == "}":
            глубина -= 1
            if глубина == 0:
                return без_комментариев[открывающая:позиция + 1]
    return без_комментариев[открывающая:]

if os.path.exists("FactoryAfter.java"):
    текст_фабрики = open("FactoryAfter.java", encoding="utf-8").read()
    блок_main = тело_метода(текст_фабрики)
    if re.search(r'\b(if|switch)\b', блок_main):
        замечания.append("в FactoryAfter.java метод main содержит ветвление по типу")
    if текст_фабрики.count("class ") < 5:
        замечания.append("в FactoryAfter.java меньше пяти классов: продукты, создатели и четвёртый тип, вероятно, не описаны")
if os.path.exists("ObserverAfter.java"):
    текст_наблюдателя = open("ObserverAfter.java", encoding="utf-8").read()
    for требуемое in ("addListener", "removeListener"):
        if требуемое not in текст_наблюдателя:
            замечания.append("в ObserverAfter.java нет метода %s" % требуемое)
отсутствующие_диаграммы = [d for d in ДИАГРАММЫ if not os.path.exists(d)]

строки = [
    "### Отчёт самопроверки «Песочница паттернов GoF»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s, вариант %d |" % (
        ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА", Номер_варианта),
    "| Дата и время отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "| Версия Java | %s |" % next(
        (s for s in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
         if "version" in s), "не определена"),
    "",
    "| Программа | Файл | Результат |",
    "|---|---|---|",
]
for название, файл, успех, пояснение in результаты:
    строки.append("| %s | `%s` | %s — %s |" % (название, файл, "пройдено" if успех else "НЕ ПРОЙДЕНО", пояснение))
строки += [
    "",
    "| Диаграммы | %s |" % ("все три на месте" if not отсутствующие_диаграммы
                            else "отсутствуют: " + ", ".join(отсутствующие_диаграммы)),
    "| Таблица и выводы | %s |" % ("заполнены" if ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
    "| Замечания | %s |" % ("нет" if not замечания else "; ".join(замечания)),
]
print("\n".join(строки))
print("\nСкопируйте отчёт выше в описание pull request.\n")

ошибки = []
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
ошибки += ["%s: %s" % (название, пояснение) for название, файл, успех, пояснение in результаты if not успех]
if отсутствующие_диаграммы:
    ошибки.append("нет диаграмм: " + ", ".join(отсутствующие_диаграммы))
if not ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены итоговая таблица и выводы")
ошибки += замечания

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")


---

## 9. Критерии оценивания

| Критерий | Баллов |
|---|---|
| Образец запущен, вопросы п. 2.5 отвечены содержательно | 10 |
| Factory Method: корректная структура паттерна, в `main` нет разбора типов | 20 |
| Factory Method: четвёртый тип добавлен новым классом | 10 |
| Observer: корректная структура, работают подписка и отписка | 20 |
| Observer: четвёртый наблюдатель добавлен без правки субъекта | 10 |
| Проверки в каждом файле осмысленны (не `[OK]` ради `[OK]`) | 10 |
| Три диаграммы классов соответствуют коду | 10 |
| Итоговая таблица, контрольные вопросы, выводы | 10 |
| **Итого** | **100** |

Снижение: задача решена не на своём варианте — минус 20; код «после» повторяет разбор типа условным оператором внутри нового класса — минус 20; диаграмма не соответствует коду — минус 10.

## 10. Для желающих (сверх задания)

| Что сделать | Зачем |
|---|---|
| Подключить настоящий JUnit 5 вместо метода `check`: `!wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.10.2/junit-platform-console-standalone-1.10.2.jar` и запуск через `java -jar` | переход к инструменту, который используется в ЛР по модульному тестированию |
| Заменить три `if` в выборе стратегии на `java.util.Map` с ключом-перечислением (`enum`) | показать, что каталог стратегий сам может быть устранён из вызывающего кода |
| Реализовать Observer на `java.util.function.Consumer` вместо своего интерфейса | сравнить классический GoF и функциональный стиль |